In [1]:
import torch
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchmetrics.detection.mean_ap import MeanAveragePrecision
from torch.utils.data import DataLoader
from torchvision import transforms

from ultralytics import YOLO
from PIL import Image

import numpy as np
import math
import cv2 as cv
import matplotlib.pyplot as plt

import os

In [2]:
class CustomDataset(Dataset):
    def __init__(self, root_dir, split, transform=None):
        self.img_dir = os.path.join(root_dir, split, 'images')
        self.label_dir = os.path.join(root_dir, split, 'labels')
        self.imgs = sorted(os.listdir(self.img_dir))
        self.transform = transform

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, index):
        img_path = os.path.join(self.img_dir, self.imgs[index])
        label_path = os.path.join(self.label_dir, os.path.splitext(self.imgs[index])[0] + '.txt')

        img = Image.open(img_path).convert('RGB')
        w, h = img.size

        boxes, labels = [], []
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    cls, x_center, y_center, box_w, box_h = map(float, line.strip().split())
                    x_min = (x_center - box_w/2) * w
                    x_max = (x_center + box_w/2) * w
                    y_min = (y_center - box_h/2) * h
                    y_max = (y_center + box_h/2) * h
                    boxes.append([x_min, y_min, x_max, y_max])
                    labels.append(int(cls) + 1)  # +1 потому что фон = 0

        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
        }

        if self.transform:
            img = self.transform(img)

        return img, target

In [3]:
transform = transforms.Compose([
    transforms.ToTensor(),  
])

train_dataset = CustomDataset('data', 'train', transform=transform)
val_dataset = CustomDataset('data', 'valid', transform=transform)

In [4]:
img, label = train_dataset[0]

In [5]:
img

tensor([[[0.9059, 0.9608, 0.8745,  ..., 0.7686, 0.7686, 0.7686],
         [0.9608, 0.8745, 1.0000,  ..., 0.7686, 0.7686, 0.7686],
         [0.8902, 0.8980, 0.9490,  ..., 0.7686, 0.7686, 0.7686],
         ...,
         [0.9647, 0.9647, 0.9647,  ..., 0.7529, 0.7529, 0.7529],
         [0.9647, 0.9647, 0.9647,  ..., 0.7490, 0.7529, 0.7529],
         [0.9647, 0.9647, 0.9647,  ..., 0.7490, 0.7529, 0.7529]],

        [[0.9059, 0.9608, 0.8745,  ..., 0.8078, 0.8078, 0.8078],
         [0.9608, 0.8745, 1.0000,  ..., 0.8078, 0.8078, 0.8078],
         [0.8902, 0.8980, 0.9490,  ..., 0.8078, 0.8078, 0.8078],
         ...,
         [0.9412, 0.9412, 0.9412,  ..., 0.7765, 0.7765, 0.7765],
         [0.9412, 0.9412, 0.9412,  ..., 0.7725, 0.7765, 0.7765],
         [0.9412, 0.9412, 0.9412,  ..., 0.7725, 0.7765, 0.7765]],

        [[0.9059, 0.9608, 0.8745,  ..., 0.8431, 0.8431, 0.8431],
         [0.9608, 0.8745, 1.0000,  ..., 0.8431, 0.8431, 0.8431],
         [0.8902, 0.8980, 0.9490,  ..., 0.8471, 0.8471, 0.

In [16]:
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(
    train_dataset[16],
    batch_size=1,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    val_dataset[16],
    batch_size=1,
    shuffle=False,
    num_workers=8,
    pin_memory=True,
    collate_fn=collate_fn
)

In [17]:
train_dataset[10]

(tensor([[[1.0000, 1.0000, 1.0000,  ..., 0.0157, 0.0157, 0.0157],
          [1.0000, 1.0000, 1.0000,  ..., 0.0157, 0.0157, 0.0157],
          [1.0000, 1.0000, 1.0000,  ..., 0.0157, 0.0157, 0.0157],
          ...,
          [0.1333, 0.1412, 0.1412,  ..., 0.2392, 0.2431, 0.2431],
          [0.1020, 0.1137, 0.1216,  ..., 0.2157, 0.2157, 0.2157],
          [0.0824, 0.0980, 0.1098,  ..., 0.1961, 0.1961, 0.1961]],
 
         [[1.0000, 1.0000, 1.0000,  ..., 0.0157, 0.0157, 0.0157],
          [1.0000, 1.0000, 1.0000,  ..., 0.0157, 0.0157, 0.0157],
          [1.0000, 1.0000, 1.0000,  ..., 0.0157, 0.0157, 0.0157],
          ...,
          [0.2275, 0.2353, 0.2353,  ..., 0.2471, 0.2510, 0.2510],
          [0.1961, 0.2078, 0.2157,  ..., 0.2235, 0.2235, 0.2235],
          [0.1765, 0.1922, 0.2000,  ..., 0.2039, 0.2039, 0.2039]],
 
         [[1.0000, 1.0000, 1.0000,  ..., 0.0235, 0.0235, 0.0235],
          [1.0000, 1.0000, 1.0000,  ..., 0.0235, 0.0235, 0.0235],
          [1.0000, 1.0000, 1.0000,  ...,

In [18]:
def get_model():
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(weights="DEFAULT")
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    num_classes = 2
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

In [19]:
device = 'cuda'
model = get_model()
model.to(device)

FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [20]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=1e-4)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.9)

epochs = 10
train_losses, val_losses, map50_scores = [], [], []
metric = MeanAveragePrecision()

for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    
    for imgs, targets in train_loader:
        imgs = [im.to(device) for im in imgs]
        tgts = [{k: v.to(device) for k, v in t.items()} for t in targets]

        loss_dict = model(imgs, tgts)
        losses = sum(loss for loss in loss_dict.values())
        train_loss += losses.item()

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    train_loss /= len(train_loader)
    train_losses.append(train_loss)

    model.eval()
    metric.reset()
    with torch.no_grad():
        for imgs, targets in valid_loader:
            imgs = [im.to(device) for im in imgs]
            tgts = [{k: v.to(device) for k, v in t.items()} for t in targets]

            preds = model(imgs)
            metric.update(preds, tgts)

    metrics = metric.compute()
    map50 = metrics["map_50"].item()
    map50_scores.append(map50)

    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f}, mAP50={map50:.4f}")

    lr_scheduler.step()

RuntimeError: DataLoader worker (pid(s) 11140, 21344, 30684, 24064, 20008, 12052, 28828, 6196) exited unexpectedly

In [ ]:
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Val Loss")
plt.legend()
plt.show()

plt.plot(map50_scores, label="mAP@0.5")
plt.legend()
plt.show()
